# Storage report examples

In [ ]:
!pip install itables

import json

with open('config.json', 'r') as config_file:
    CONFIG = json.load(config_file)

In [ ]:
# Set and echo back our environment variables

CONSORTIUM_NAME = CONFIG["CONSORTIUM_NAME"] # SET CONSORTIUM NAME in .env file
CONS_REPORT_URL = CONFIG["CONS_REPORT_URL"]
VOL_REPORT_URL = CONFIG["VOL_REPORT_URL"]
SERV_REPORT_URL = CONFIG["SERV_REPORT_URL"]
SYSTEM_CAPACITY_TB = CONFIG["SYSTEM_CAPACITY_TB"]

# print required CONFIG values so we can inspect
print("CONSORTIUM_NAME: {0}".format(CONSORTIUM_NAME))
print("CONS_REPORT_URL: {0}".format(CONS_REPORT_URL))
print("VOL_REPORT_URL: {0}".format(VOL_REPORT_URL))
print("SERV_REPORT_URL: {0}".format(SERV_REPORT_URL))
print("SYSTEM_CAPACITY_TB: {0}".format(SYSTEM_CAPACITY_TB))


In [ ]:
TIMEOUT_SEC = 500 # need long timeout for some of the reports to load.
# Please try again later if you get a 500 errror.

import socket
socket.setdefaulttimeout(TIMEOUT_SEC)

import pandas as pd
import ssl  
ssl._create_default_https_context = ssl._create_unverified_context

# interactive tables that we can size and sort
from itables import init_notebook_mode
init_notebook_mode(all_interactive=True)

import matplotlib.pyplot as plt

# set system overall capacities


df_cons = pd.read_csv(CONFIG["CONS_REPORT_URL"], index_col='Consortia')
df_vols = pd.read_csv(CONFIG["VOL_REPORT_URL"])
df_serv = pd.read_csv(CONFIG["SERV_REPORT_URL"])

In [ ]:
df_cons

## Consortium summary

Used, provisioned and quota for each storage type

In [ ]:
storage_types = ['SOF', 'PFS', 'SSD', 'SSDE', 'HPOS', 'Tape']
colours = ['tab:blue','tab:orange','tab:green', 'tab:red']

row = df_cons.loc[CONSORTIUM_NAME]

# show values in table
row


In [ ]:
# plot summary

used = [row["usedSOF_TB"], row["usedPFS_TB"], row["usedSSD_TB"], row["usedSSDE_TB"], row["usedHPOS_TB"], row["usedTape_TB"]]
#req = [row["reqSOF_TB"], row["reqPFS_TB"], row["reqSSD_TB"], row["reqHPOS_TB"], row["reqTape_TB"]]
prov = [row["provSOF_TB"], row["provPFS_TB"], row["provSSD_TB"], row["provSSDE_TB"], row["provHPOS_TB"], row["provTape_TB"]]
quota = [row["quotaSOF_TB"], row["quotaPFS_TB"], row["quotaSSD_TB"], row["quotaSSDE_TB"], row["quotaHPOS_TB"], row["quotaTape_TB"]]
df_row = pd.DataFrame({'used': used, 'prov': prov, 'quota': quota}, index=storage_types)    #'req': req, 
ax = df_row.plot.bar(color=colours)
ax.set(ylabel='TB', title="Used, prov & quota by storage type for %s" % CONSORTIUM_NAME)


## GWS Vols sorted by provisioned size

In [ ]:
# All GWS volumes start with /gws
df_gws = df_vols.query(
    'StorageType=="ssde" & \
    Path.str.startswith("/gws") & \
    Consortia=="%s"' % CONSORTIUM_NAME
)

In [ ]:
df_gws

In [ ]:
### Plot GWS volumes (prov, used) in descending order of (prov-used)

df_subset = (
    df_gws.sort_values(
        by=["Quota/Provisioned(OnSystem)TB"],
        ascending=True))[[
            "Path",
            "UsedTB",
            "Required/Requested(InPP)TB",
            "Quota/Provisioned(OnSystem)TB"
        ]]

In [ ]:
df_subset

In [ ]:
ax = df_subset.plot.barh(
    x="Path",
    figsize=(10,50),
    xlabel='TB',
    ylabel='volume',
    title='Volume used / required / provisioned',
    color = ['tab:blue','tab:orange','tab:green', 'tab:red']
)

### Most empty space

Define a metric "proportion_empty" as:
empty = (provisioned - used) / provisioned

Then sort the list to show the largest volumes with the most empty space

In [ ]:
df_subset["proportion_empty"] = (df_subset["Quota/Provisioned(OnSystem)TB"] - df_subset["UsedTB"])/df_subset["Quota/Provisioned(OnSystem)TB"]

In [ ]:
df_empty = (
    df_subset.sort_values(
        by=["proportion_empty","Quota/Provisioned(OnSystem)TB"],
        ascending=False))[[
            "Path",
            "proportion_empty",
            "UsedTB",
            "Quota/Provisioned(OnSystem)TB",
        ]]
df_empty

### GWS services with less than 2 users

In [ ]:
df_serv_lt2u = df_serv[df_serv['UserCount'] < 2]
df_serv_lt2u.sort_values(by=["UserCount"])

# Report by heat

- show volumes with files unaccessed for
```-1h	1h-1d	1d-1w	1w-1m	1m-3m	3m-6m	6m-1y	1y-2y	2y-5y	5y-*```

In [ ]:
#df_heat = pd.read_csv("gws_heats_.csv")
#df_heat.sort_values(by=["5y-*","2y-5y","1y-2y"],ascending=False)